In [124]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {device}")

if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU index: {torch.cuda.current_device()}")
    print(f"Current GPU name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU {i} total VRAM: {total_vram_gb:.2f} GB")

    current = torch.cuda.current_device()
    allocated_gb = torch.cuda.memory_allocated(current) / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(current) / (1024 ** 3)
    print(f"GPU {current} allocated VRAM: {allocated_gb:.2f} GB")
    print(f"GPU {current} reserved VRAM: {reserved_gb:.2f} GB")

CUDA available: True
Selected device: cuda
GPU count: 1
Current GPU index: 0
Current GPU name: NVIDIA GeForce GTX 1650 Ti
GPU 0 total VRAM: 3.63 GB
GPU 0 allocated VRAM: 0.00 GB
GPU 0 reserved VRAM: 0.02 GB


In [107]:
import csv

def load_csv_data(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append(row)
    return data

def format_input(entry):
    return (
        f"### Instruction:\nConvert the following text to brainrot slang.\n\n"
        f"### Input:\n{entry['original_message']}"
    )

In [108]:
import torch

class InstructionDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['brainrot_message']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [109]:
import tiktoken
from torch.utils.data import random_split

tokenizer = tiktoken.get_encoding("gpt2")
data = load_csv_data("../data/synthetic/data.csv")

dataset = InstructionDataset(data, tokenizer)

train_size = int(0.85 * len(dataset))
val_size = int(0.05 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

print(f"Total: {len(dataset)}, Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Total: 4080, Train: 3468, Val: 204, Test: 408


In [110]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch, pad_token_id=50256, device="cpu"):
    batch_tensors = [torch.tensor(item) for item in batch]
    padded = pad_sequence(batch_tensors, batch_first=True, padding_value=pad_token_id)

    # Build a mask: True for real tokens + one trailing pad (the EOS target)
    lengths = torch.tensor([len(item) for item in batch])

    # positions beyond length+1 are pure padding to ignore

    positions = torch.arange(padded.size(1)).unsqueeze(0)  # (1, seq_len)
    
    target_mask = positions >= lengths.unsqueeze(1)  # True for padding positions

    targets_tensor = torch.roll(padded, -1, dims=1).clone()

    # Set last real token's target to pad_token_id (EOS)
    targets_tensor[torch.arange(len(batch)), lengths - 1] = pad_token_id

    # Mask only pure padding positions
    targets_tensor[target_mask] = -100

    return padded.to(device), targets_tensor.to(device)

In [111]:
batch = (
    [1,2,3,4,5,6],
    [7,8,9],
    [10,11]
)
inputs, targets = custom_collate(batch)
print(inputs)
print(targets)

tensor([[    1,     2,     3,     4,     5,     6],
        [    7,     8,     9, 50256, 50256, 50256],
        [   10,    11, 50256, 50256, 50256, 50256]])
tensor([[    2,     3,     4,     5,     6, 50256],
        [    8,     9, 50256,  -100,  -100,  -100],
        [   11, 50256,  -100,  -100,  -100,  -100]])


In [112]:
inputs, targets = custom_collate(dataset, device=device)
print(inputs)
print(targets)

tensor([[21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        ...,
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256]], device='cuda:0')
tensor([[46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        ...,
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100]], device='cuda:0')


In [113]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate
)
val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
    collate_fn=custom_collate
)
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
    collate_fn=custom_collate
)

In [123]:
for batch in train_loader:
    inputs, targets = batch
    print(f"Batch input shape: {inputs.shape}, Batch target shape: {targets.shape}")
    break

Batch input shape: torch.Size([8, 243]), Batch target shape: torch.Size([8, 243])
